In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
import json
import pandas as pd
import numpy as np
from tqdm import tqdm

In [6]:
BASE_PATH = "/content/drive/MyDrive/PrismHire/data/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge"

jsonl_path = f"{BASE_PATH}/candidates.jsonl"

In [9]:
candidates = []

with open(jsonl_path, "r") as f:
    for line in tqdm(f):
        candidates.append(json.loads(line))

100000it [00:21, 4640.01it/s]


In [10]:
len(candidates)

100000

# =====================================
# Feature Extraction Function
# =====================================

In [14]:
def extract_candidate_features(candidate):
    row = {}

    return row

In [15]:
def extract_candidate_features(candidate):

    row = {}

    profile = candidate["profile"]

    row["candidate_id"] = candidate["candidate_id"]

    row["years_experience"] = profile["years_of_experience"]

    row["current_title"] = profile["current_title"]

    row["current_industry"] = profile["current_industry"]

    row["current_company_size"] = profile["current_company_size"]

    row["country"] = profile["country"]

    return row

In [17]:
sample = extract_candidate_features(candidates[4])

sample

{'candidate_id': 'CAND_0000005',
 'years_experience': 11.0,
 'current_title': 'Accountant',
 'current_industry': 'Manufacturing',
 'current_company_size': '1001-5000',
 'country': 'India'}

In [25]:
def extract_candidate_features(candidate):

    row = {}

    # =========================
    # Extract Sections
    # =========================
    profile = candidate["profile"]
    skills = candidate["skills"]
    career = candidate["career_history"]
    signals = candidate["redrob_signals"]

    # =========================
    # Basic Profile Features
    # =========================
    row["candidate_id"] = candidate["candidate_id"]
    row["years_experience"] = profile["years_of_experience"]
    row["current_title"] = profile["current_title"]
    row["current_industry"] = profile["current_industry"]
    row["current_company_size"] = profile["current_company_size"]
    row["country"] = profile["country"]

    # =========================
    # Skill Features
    # =========================
    row["num_skills"] = len(skills)

    skill_map = {
        "beginner": 1,
        "intermediate": 2,
        "advanced": 3,
        "expert": 4
    }

    proficiency_scores = []
    endorsements = []
    skill_durations = []

    for s in skills:
        proficiency_scores.append(
            skill_map.get(s["proficiency"], 0)
        )
        endorsements.append(
            s.get("endorsements", 0)
        )
        skill_durations.append(
            s.get("duration_months", 0)
        )

    row["avg_skill_proficiency"] = (
        np.mean(proficiency_scores)
        if proficiency_scores else 0
    )

    row["avg_skill_endorsements"] = (
        np.mean(endorsements)
        if endorsements else 0
    )

    row["avg_skill_duration"] = (
        np.mean(skill_durations)
        if skill_durations else 0
    )

    # =========================
    # Career Features
    # =========================
    row["num_jobs"] = len(career)

    job_durations = []
    industries = set()
    company_sizes = set()

    leadership_titles = [
        "manager",
        "lead",
        "head",
        "director",
        "principal",
        "architect"
    ]

    leadership_score = 0

    for job in career:

        job_durations.append(
            job.get("duration_months", 0)
        )

        industries.add(
            job.get("industry", "Unknown")
        )

        company_sizes.add(
            job.get("company_size", "Unknown")
        )

        title = job.get("title", "").lower()

        for word in leadership_titles:
            if word in title:
                leadership_score += 1
                break

    row["avg_job_duration"] = (
        np.mean(job_durations)
        if job_durations else 0
    )

    row["max_job_duration"] = (
        max(job_durations)
        if job_durations else 0
    )

    row["industry_exposure"] = len(industries)

    row["company_scale_exposure"] = len(
        company_sizes
    )

    row["leadership_experience"] = (
        leadership_score
    )

    # =========================
    # Redrob Recruitability Signals
    # =========================
    row["open_to_work"] = int(
        signals["open_to_work_flag"]
    )

    row["profile_completeness"] = (
        signals["profile_completeness_score"]
    )

    row["recruiter_response_rate"] = (
        signals["recruiter_response_rate"]
    )

    row["avg_response_time_hours"] = (
        signals["avg_response_time_hours"]
    )

    row["profile_views_30d"] = (
        signals["profile_views_received_30d"]
    )

    row["search_appearances"] = (
        signals["search_appearance_30d"]
    )

    row["saved_by_recruiters"] = (
        signals["saved_by_recruiters_30d"]
    )

    row["interview_completion_rate"] = (
        signals["interview_completion_rate"]
    )

    # Offer Acceptance
    offer_rate = signals[
        "offer_acceptance_rate"
    ]

    if offer_rate == -1:
        offer_rate = 0

    row["offer_acceptance_rate"] = (
        offer_rate
    )

    # GitHub Activity
    github_score = signals[
        "github_activity_score"
    ]

    if github_score == -1:
        github_score = 0

    row["github_activity_score"] = (
        github_score
    )

    row["connection_count"] = (
        signals["connection_count"]
    )

    row["endorsements_received"] = (
        signals["endorsements_received"]
    )

    row["notice_period_days"] = (
        signals["notice_period_days"]
    )

    row["willing_to_relocate"] = int(
        signals["willing_to_relocate"]
    )

    # Work Mode Encoding
    work_mode_map = {
        "remote": 1,
        "hybrid": 2,
        "onsite": 3,
        "flexible": 4
    }

    row["preferred_work_mode"] = (
        work_mode_map.get(
            signals["preferred_work_mode"],
            0
        )
    )

    row["verified_email"] = int(
        signals["verified_email"]
    )

    row["verified_phone"] = int(
        signals["verified_phone"]
    )

    row["linkedin_connected"] = int(
        signals["linkedin_connected"]
    )

    # Skill Assessment Scores
    assessment_scores = list(
        signals["skill_assessment_scores"].values()
    )

    row["avg_assessment_score"] = (
        np.mean(assessment_scores)
        if assessment_scores
        else 0
    )

    return row

In [27]:
sample = extract_candidate_features(candidates[0])

for k, v in sample.items():
    print(f"{k}: {v}")

candidate_id: CAND_0000001
years_experience: 6.9
current_title: Backend Engineer
current_industry: IT Services
current_company_size: 10001+
country: Canada
num_skills: 17
avg_skill_proficiency: 2.235294117647059
avg_skill_endorsements: 17.176470588235293
avg_skill_duration: 24.823529411764707
num_jobs: 2
avg_job_duration: 41.0
max_job_duration: 55
industry_exposure: 2
company_scale_exposure: 2
leadership_experience: 0
open_to_work: 1
profile_completeness: 86.9
recruiter_response_rate: 0.34
avg_response_time_hours: 177.8
profile_views_30d: 23
search_appearances: 249
saved_by_recruiters: 4
interview_completion_rate: 0.71
offer_acceptance_rate: 0.58
github_activity_score: 9.2
connection_count: 356
endorsements_received: 35
notice_period_days: 60
willing_to_relocate: 0
preferred_work_mode: 3
verified_email: 1
verified_phone: 1
linkedin_connected: 0
avg_assessment_score: 49.724999999999994


In [28]:
rows = []

In [29]:
from tqdm import tqdm

In [30]:
rows = []

for candidate in tqdm(candidates):
    row = extract_candidate_features(
        candidate
    )
    rows.append(row)

100%|██████████| 100000/100000 [00:13<00:00, 7248.04it/s]


In [31]:
master_df = pd.DataFrame(rows)

In [32]:
master_df.shape

(100000, 35)

In [33]:
master_df.head()

,candidate_id,years_experience,current_title,current_industry,current_company_size,country,num_skills,avg_skill_proficiency,avg_skill_endorsements,avg_skill_duration,...,github_activity_score,connection_count,endorsements_received,notice_period_days,willing_to_relocate,preferred_work_mode,verified_email,verified_phone,linkedin_connected,avg_assessment_score
0,CAND_0000001,6.9,Backend Engineer,IT Services,10001+,Canada,17,2.235294,17.176471,24.823529,...,9.2,356,35,60,0,3,1,1,0,49.725
1,CAND_0000002,12.5,Operations Manager,IT Services,10001+,India,9,1.666667,7.777778,20.333333,...,0.0,179,3,60,0,4,0,0,0,0.000
2,CAND_0000003,1.1,Customer Support,IT Services,10001+,USA,6,1.500000,7.833333,17.666667,...,0.0,19,46,150,0,2,1,0,0,0.000
3,CAND_0000004,3.8,Marketing Manager,Paper Products,201-500,Australia,10,1.500000,8.000000,13.400000,...,0.0,485,22,120,1,3,1,1,1,0.000
4,CAND_0000005,11.0,Accountant,Manufacturing,1001-5000,India,6,1.666667,14.666667,24.500000,...,0.0,300,14,30,1,2,0,1,1,0.000


In [34]:
master_df.isnull().sum()

,0
candidate_id,0
years_experience,0
current_title,0
current_industry,0
current_company_size,0
country,0
num_skills,0
avg_skill_proficiency,0
avg_skill_endorsements,0
avg_skill_duration,0


In [37]:
master_df.to_parquet(
    "/content/drive/MyDrive/PrismHire/master_candidates.parquet",
    index=False
)

In [36]:
master_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 35 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   candidate_id               100000 non-null  object 
 1   years_experience           100000 non-null  float64
 2   current_title              100000 non-null  object 
 3   current_industry           100000 non-null  object 
 4   current_company_size       100000 non-null  object 
 5   country                    100000 non-null  object 
 6   num_skills                 100000 non-null  int64  
 7   avg_skill_proficiency      100000 non-null  float64
 8   avg_skill_endorsements     100000 non-null  float64
 9   avg_skill_duration         100000 non-null  float64
 10  num_jobs                   100000 non-null  int64  
 11  avg_job_duration           100000 non-null  float64
 12  max_job_duration           100000 non-null  int64  
 13  industry_exposure          100

In [38]:
master_df.to_csv(
    "/content/drive/MyDrive/PrismHire/master_candidates.csv",
    index=False
)

In [39]:
!ls "/content/drive/MyDrive/PrismHire/"

data  graphs		     master_candidates.parquet	notebooks
docs  master_candidates.csv  models			outputs
